# SepsisGuard-UAR — Frozen predictor and evidence agents

Choose your **A100 80GB / high-RAM runtime**, then **Runtime → Run all**. Approve the private Drive mount when prompted. The pinned BF16 Qwen profile checks GPU capacity before downloading weights.

This notebook restores your completed LSTM and preparation checkpoints. It fits a sigmoid calibrator on the reserved **A/calibration** data, snapshots public CDC/NIH reference text, and runs a local **Qwen3.8-27B / LangGraph** retrieval, drafting and verification loop over three deterministic A/validation endpoints. It preserves the trained predictor and keeps B raw files locked.

The model uses one shared native Transformers instance, BF16, an 8K total context cap and bounded role outputs. Each case permits at most six model decisions and two local retrieval attempts. If citations, structured output or evidence checks fail, the agent abstains. A same-model verifier is not independent factual certification.

First use downloads roughly 54–56GB of base weights onto runtime storage; Drive holds reference snapshots, calibration, case traces and result files. Existing trained models are verified by hash and are not retrained. Completed cases resume using the same source, inputs, model revision and corpus. The inference budget is 20 minutes after model loading.

Read `AGENT_RESULT.json` and `AGENT_REPORT.md` under `MyDrive/SepsisGuard-UAR/agents/<run-id>`. This is a small functional research run, not a clinical validation or a benchmark of explanation correctness. No diagnosis or treatment recommendation is produced.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, uuid
from google.colab import drive

REPO_URL = 'https://github.com/shakiful/Sepsisguard-Uar.git'
PROJECT = Path('/content/sepsisguard-uar-unified')
# Keep previous notebooks and downloaded records intact.
if PROJECT.exists():
    assert (PROJECT/'.git').is_dir(), 'The chosen source folder is not a Git checkout.'
    remote = subprocess.check_output(['git','-C',str(PROJECT),'remote','get-url','origin'],text=True).strip()
    assert remote.lower().rstrip('/').removesuffix('.git') == REPO_URL.lower().removesuffix('.git'), 'Unexpected source repository.'
    dirty = subprocess.check_output(['git','-C',str(PROJECT),'status','--porcelain','--untracked-files=no'],text=True).strip()
    if dirty:
        PROJECT = Path('/content/sepsisguard-uar-unified-'+uuid.uuid4().hex[:8])
if not PROJECT.exists():
    subprocess.run(['git','clone','--branch','main','--single-branch',REPO_URL,str(PROJECT)],check=True)
else:
    subprocess.run(['git','-C',str(PROJECT),'fetch','origin','main'],check=True)
    subprocess.run(['git','-C',str(PROJECT),'merge','--ff-only','origin/main'],check=True)
os.chdir(PROJECT)
compute = json.loads(subprocess.check_output([sys.executable,'scripts/compute_check.py'],text=True))
print(json.dumps(compute,indent=2))
assert compute['cuda_available'] and compute['gpus'][0]['vram_gb'] >= 70, 'Select the authorized A100 80GB runtime, then Run all. No Qwen weights have been downloaded.'
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/SepsisGuard-UAR')
DRIVE_ROOT.mkdir(parents=True,exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[dev,download,agents]'],check=True)
old_data = Path('/content/sepsisguard-uar/data/raw')
DATA_ROOT = old_data if old_data.is_dir() else PROJECT/'data/raw'
print('Source commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
print('Persistent project folder:',DRIVE_ROOT)


In [ ]:
# Use the reviewed frozen predictor in configs/agent_profile.json.
AGENT_CASES = 3
MAX_INFERENCE_MINUTES = 20
command = [sys.executable,'-u','-m','scripts.run_agents',
           '--data-root',str(DATA_ROOT),'--storage-root',str(DRIVE_ROOT),
           '--profile','configs/agent_profile.json','--cases',str(AGENT_CASES),
           '--max-minutes',str(MAX_INFERENCE_MINUTES)]
process = subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in process.stdout:
    print(line,end='',flush=True)
if process.wait():
    raise RuntimeError('Agent run stopped. Review the error above; trained models and verified case checkpoints remain available.')
print('Agent workflow finished. Inspect AGENT_RESULT.json, AGENT_REPORT.md and case traces; abstentions and verifier decisions require review.')
